# P06 — Prédiction du prix des carburants à la pompe

**Contexte métier.** Applications de comparaison de prix (essence.app-type), flottes de véhicules, commission européenne de la concurrence : expliquer les écarts de prix entre stations voisines (même carburant, même jour, même département : 20 à 30 c/L d'écart). Enjeu : pouvoir d'achat, détection de zones captives.

**Objectif de l'EDA.** Identifier la cible (prix du carburant) et les régresseurs (type de carburant, département, enseigne, automatisation, services, taille de la commune).

**Données.** Prix des carburants en France — flux quotidien, Ministère de l'Économie (data.economie.gouv.fr). Licence ouverte Etalab 2.0. 75 306 lignes = 9 946 stations × carburants ; une ligne = prix d'un carburant dans une station (instantané du jour).

## 1. Téléchargement

Instantané quotidien des prix via l'API de data.economie.gouv.fr :

In [ ]:
import urllib.request, os
os.makedirs("data", exist_ok=True)

URL = "https://data.economie.gouv.fr/api/explore/v2.1/catalog/datasets/prix-carburants-quotidien/exports/csv?use_labels=true"
if not os.path.exists("data/prix-carburants-quotidien.csv"):
    urllib.request.urlretrieve(URL, "data/prix-carburants-quotidien.csv")
print(os.path.getsize("data/prix-carburants-quotidien.csv") / 1e6, "Mo")

## 2. Chargement

Une ligne = station × carburant. La colonne `Prix` est le prix actuel en €/L ; `Carburant` le type ; les colonnes département, région, automate et services décrivent la station.

In [ ]:
import pandas as pd, numpy as np

keep = ["Prix", "Carburant", "prix_id", "Mise à jour des prix", "pop", "ville",
        "Région", "Numéro Département", "Département", "Services proposés",
        "Carburant en rupture", "Automate 24-24 (oui/non)", "id"]
df = pd.read_csv("data/prix-carburants-quotidien.csv", sep=";", usecols=keep, low_memory=False)
df.columns = [c.strip() for c in df.columns]
df["Prix"] = pd.to_numeric(df["Prix"], errors="coerce")
df["n_services"] = df["Services proposés"].fillna("").str.split(",").apply(len)
print(len(df), "lignes,", df["id"].nunique(), "stations")
print(df["Carburant"].value_counts().to_dict())

## 3. Analyse

Distribution des prix par carburant, écarts de prix par département (Gazole), effet de l'automate, de la ruralité (`pop` R/A) et du nombre de services.

In [ ]:
print("Prix global :", df["Prix"].describe(percentiles=[.1, .5, .9]).round(3).to_dict())
print()
print("Prix médian par carburant :", df.groupby("Carburant")["Prix"].median().round(3).to_dict())
gz = df[df["Carburant"] == "Gazole"].dropna(subset=["Prix"])
print("Gazole par département (top) :", gz.groupby("Numéro Département")["Prix"].median().round(3).sort_values(ascending=False).head(5).to_dict())
print("Gazole par département (bas) :", gz.groupby("Numéro Département")["Prix"].median().round(3).sort_values().head(5).to_dict())
print()
print("Prix médian automate 24/24 :", df.groupby("Automate 24-24 (oui/non)")["Prix"].median().round(3).to_dict())
print("Prix médian par type de commune :", df.groupby("pop")["Prix"].median().round(3).to_dict())
num = gz.assign(automate=(gz["Automate 24-24 (oui/non)"] == "Oui").astype(int))
print("Corr. Spearman (Gazole) :", num[["Prix", "n_services", "automate"]].corr("spearman")["Prix"].round(2).to_dict())

## 4. Résultats observés (exécution du 16/10/2026)

- n lignes : 75306
- n stations : 9946
- carburants : Gazole : 22992, E10 : 16666, SP98 : 15869, E85 : 7198, SP95 : 6927, GPLc : 2430
- prix prix stats : count : 72 082.00, mean : 2.09, std : 0.47, min : 0.60, 10% : 1.00, 25% : 2.15, 50% : 2.25, 75% : 2.34, 90% : 2.40, m…
- prix median par carburant : E10 : 2.18, E85 : 0.90, GPLc : 1.05, Gazole : 2.36, SP95 : 2.24, SP98 : 2.30
- gazole median par dept top8 : 67 : 2.50, 68 : 2.46, 90 : 2.45, 57 : 2.42, 52 : 2.41, 88 : 2.41, 25 : 2.40, 21 : 2.40
- gazole median par dept bottom5 : 2B : 2.25, 2A : 2.25, 94 : 2.25, 95 : 2.25, 92 : 2.25
- ecart sp95 gazole mediane : -0.11
- prix median automate oui non : Non : 2.25, Oui : 2.25
- prix median par pop : A : 2.28, R : 2.25
- prix median par region : Auvergne-Rhône-Alpes : 2.25, Bourgogne-Franche-Comté : 2.30, Bretagne : 2.25, Centre-Val de Loire : 2.25, Corse : 2.2…
- pct en rupture : 96.20
- pct prix recents 7j : 71.20
- prix gazole corr spearman num : Prix : 1.00, n_services : -0.21, automate : 0.04, dept : 0.01

## 5. Variable cible et régresseurs

**Cible (régression) : `Prix` en €/L pour un couple (station, carburant).** Variante métier : classification "station plus chère que la médiane de son département" (détection de zones captives).

**Régresseurs retenus :**
- `Carburant` : effet massif (Gazole 2,358 € médian, E85 0,899 €) — à contrôler systématiquement, ou modéliser un carburant à la fois (Gazole)
- `Numéro Département` / `Département` / `Région` : écarts géographiques persistants
- `pop` (commune Rurale/urbaine), `Automate 24-24`, `n_services` (le nombre de services proposés), `Carburant en rupture`
- Fraîcheur : `Mise à jour des prix` (poids les prix anciens)

**Pièges :**
- C'est un instantané : pas de dimension temporelle dans ce fichier. Pour prévoir un prix, il faut l'historique des flux quotidiens (archiver chaque jour ou utiliser l'historique HeRÉA/odr)
- 3 224 lignes sans prix (stations qui ne pratiquent plus le carburant ou rupture) : à écarter
- Le prix dépend d'abord du carburant : toujours segmenter ou inclure le type en feature, sinon R² artificiellement élevé
- Certaines "prix_id" anciens (2014) : vérifier la fraîcheur avant toute carte ou moyenne